# Raman Workflow Tutorial with VASP

## Background
The Raman workflow computes Raman tensors from finite differences of the high-frequency dielectric tensor along the Gamma-point normal modes, as used for the Computational Raman Database ([Bagheri and Komsa, Sci. Data 10, 80 (2023)](https://doi.org/10.1038/s41597-023-01988-5)).

The flow consists of
1. the phonon workflow: tight relaxation, finite displacement phonons with Phonopy and (optionally) Born charges,
2. the Gamma-point normal modes of the primitive cell and the selection of the Raman-active modes from the irreducible representations,
3. two LEPSILON calculations per selected mode, displaced by $\pm u$ along the mode,
4. the Raman tensors $R_k = \frac{V}{4\pi}\frac{\varepsilon(+u)-\varepsilon(-u)}{2u}\left|\frac{\mathbf{e}_k}{\sqrt{m}}\right|$, the activities $45\alpha^2 + 7\beta^2$ and a broadened spectrum.

## Let's build the workflow
We start from the primitive cell of silicon, which has a single triply degenerate Raman-active mode ($T_{2g}$).

In [ ]:
import warnings

from pymatgen.core import Structure

from atomate2.vasp.flows.phonons import PhononMaker
from atomate2.vasp.flows.raman import RamanMaker

warnings.filterwarnings("ignore")

si_structure = Structure(
    lattice=[[0, 2.73, 2.73], [2.73, 0, 2.73], [2.73, 2.73, 0]],
    species=["Si", "Si"],
    coords=[[0, 0, 0], [0.25, 0.25, 0.25]],
)

By default, ENCUT is set to 1.3 times the largest ENMAX of the POTCARs for all VASP jobs of the flow (`encut_enmax_factor=1.3`). This requires the POTCARs to be available when the flow is created (`PMG_VASP_PSP_DIR`). Here, we pass ENCUT explicitly instead. Silicon is not polar, so the Born-charge calculation of the phonon workflow is switched off.

In [ ]:
maker = RamanMaker(
    phonon_maker=PhononMaker(min_length=15.0, born_maker=None),
    step_size=0.005,
    mode_selection="raman_active",
)
flow = maker.make(si_structure, mp_id="mp-149", encut=520)

The flow contains the phonon workflow followed by the Raman jobs. The number of dielectric calculations is only known once the phonons are finished, so `raman displacements` creates them at runtime.

In [ ]:
[job.name for job, _ in flow.iterflow()]

In [ ]:
flow.draw_graph().show()

## Input of the displaced dielectric calculations
The displaced structures are calculated with `RamanDielectricSetGenerator`, which follows the phonondb `INCAR-nac` settings and uses a fully automatic Gamma-centred k-mesh. Its length parameter `Rk` is chosen at runtime from the band gap of the static calculation of the phonon workflow: `Rk = 20` above 2 eV, `Rk = 30` for 1-2 eV and `Rk = 40` below 1 eV. Here, we show the input for `Rk = 40`.

In [ ]:
from atomate2.vasp.sets.raman import RamanDielectricSetGenerator

input_set = RamanDielectricSetGenerator(
    kpoints_length=40, user_incar_settings={"ENCUT": 520}
).get_input_set(si_structure, potcar_spec=True)
input_set.incar

In [ ]:
input_set.kpoints

The settings of the dielectric calculations can be changed with the powerups by filtering on the job name `raman dielectric`. A k-point setting given this way overrides the band-gap rule.

In [ ]:
from atomate2.vasp.powerups import (
    update_user_incar_settings,
    update_user_kpoints_settings,
)

flow = update_user_incar_settings(flow, {"KPAR": 2}, name_filter="raman dielectric")
flow = update_user_kpoints_settings(
    flow, {"length": 50}, name_filter="raman dielectric"
)

## Running the workflow
Locally:
```python
from jobflow import run_locally

run_locally(flow, create_folders=True)
```

With jobflow-remote:
```python
from jobflow_remote import submit_flow

submit_flow(flow, worker="my_worker")
```

With FireWorks:
```python
from fireworks import LaunchPad
from jobflow.managers.fireworks import flow_to_workflow

LaunchPad.auto_load().add_wf(flow_to_workflow(flow))
```

## Analysing the results
The output of the flow is a `RamanDoc`. It contains the frequencies, irreducible representations and Raman activity of all modes, the Raman tensors of the computed modes and a Lorentzian-broadened spectrum, as well as the band gap and the `Rk` used for the dielectric calculations. The analysis job also writes `raman.json` in the atomate-CRD format.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from atomate2.utils.raman import lorentzian_spectrum


def plot_raman(doc: dict, width: float = 4.0) -> plt.Figure:
    """Plot the Raman spectrum of a RamanDoc dictionary."""
    modes = [m for m in doc["modes"] if m["frequency_cm"] > 0]
    freqs = np.array([m["frequency_cm"] for m in modes])
    ints = np.array([m["intensity"] for m in modes])
    ints /= ints.max()
    x, y = lorentzian_spectrum(freqs, ints, width=width)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(x, y, color="k")
    ax.vlines(freqs, 0, ints, color="tab:red")
    for m, i in zip(modes, ints, strict=True):
        if m["irrep"] and i > 0.05:
            ax.text(m["frequency_cm"], i + 0.03, m["irrep"], ha="center")
    ax.set_xlabel("Raman shift (cm$^{-1}$)")
    ax.set_ylabel("Intensity (arb. units)")
    ax.set_ylim(0, 1.15)
    return fig

After the flow has finished, the document can be loaded from the job store and plotted:
```python
from jobflow import SETTINGS

store = SETTINGS.JOB_STORE
store.connect()
result = store.query_one(
    {"name": "raman analysis"}, load=True, sort={"completed_at": -1}
)
doc = result["output"]
for m in doc["modes"]:
    print(m["mode"], m["irrep"], round(m["frequency_cm"], 1), m["raman_activity"])
plot_raman(doc).show()
```